In [1]:
from load_datasets import load_train_test_sets, split_train_evaluate, create_submission_file

In [2]:
X_train, X_test, train_target = load_train_test_sets()

In [3]:
X_train.head()

,LotFrontage,LotArea,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,1stFlrSF,...,MoSold_11,MoSold_12,MoSold_2,MoSold_3,MoSold_4,MoSold_5,MoSold_6,MoSold_7,MoSold_8,MoSold_9
0,-0.229372,-0.207142,1.050994,0.878668,0.511418,0.575425,-0.288653,-0.944591,-0.459303,-0.793434,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.451936,-0.091886,0.156734,-0.429577,-0.574410,1.171992,-0.288653,-0.641228,0.466465,0.257140,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-0.093110,0.073480,0.984752,0.830215,0.323060,0.092907,-0.288653,-0.301643,-0.313369,-0.627826,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.456474,-0.096897,-1.863632,-0.720298,-0.574410,-0.499274,-0.288653,-0.061670,-0.687324,-0.521734,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.633618,0.375148,0.951632,0.733308,1.364570,0.463568,-0.288653,-0.174865,0.199680,-0.045611,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Generate a model using Decision Trees

In [4]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

In [5]:
dt_model = DecisionTreeRegressor()

In [6]:
dec_tr_model, train_loss, val_loss = split_train_evaluate(dt_model, X_train, train_target)

Training Loss: 0.0
Validation Loss: 0.21470534473128733


In [7]:
dec_tr_preds = dec_tr_model.predict(X_test)
dec_tr_preds

array([118500., 160000., 187500., ..., 145000., 127000., 310000.],
      shape=(1459,))

In [8]:
create_submission_file(dec_tr_preds, 'decision_tree_sub')

        Id  SalePrice
0     1461   118500.0
1     1462   160000.0
2     1463   187500.0
3     1464   180000.0
4     1465   153900.0
...    ...        ...
1454  2915    66500.0
1455  2916    80000.0
1456  2917   145000.0
1457  2918   127000.0
1458  2919   310000.0

[1459 rows x 2 columns]


### Hyperparameter Tuning the Decision Tree using GridSearchCV

In [9]:
from sklearn.model_selection import GridSearchCV
import numpy as np

In [10]:
dt_model_tune = DecisionTreeRegressor(random_state=42)

In [11]:
params_grid = {
  'max_depth': [10, 15, 20, 30], 
  'min_samples_split': [2, 3, 5, 10, 15, 20, 30, 40], 
  'min_samples_leaf': [5, 10 , 15, 20, 25]
}

In [12]:
grid_search = GridSearchCV(
  estimator=dt_model_tune, 
  param_grid=params_grid, 
  cv=4, 
  scoring='neg_root_mean_squared_log_error', 
  n_jobs=-1
)

In [13]:
grid_search.fit(X_train, train_target)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [10, 15, ...], 'min_samples_leaf': [5, 10, ...], 'min_samples_split': [2, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_log_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",4
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verb

In [14]:
grid_search.best_params_

{'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 30}

In [15]:
grid_search.best_score_

np.float64(-0.19755016890293492)

In [16]:
best_dec_tr_model = grid_search.best_estimator_

tr_preds = best_dec_tr_model.predict(X_train)
tr_preds

array([201034.61538462, 184663.63636364, 212047.69230769, ...,
       279807.75      , 117947.95833333, 175225.4       ], shape=(1460,))

In [17]:
test_tr_preds = best_dec_tr_model.predict(X_test)

create_submission_file(test_tr_preds, 'decision_tree_tuned_sub')

        Id      SalePrice
0     1461  109430.000000
1     1462  142169.736842
2     1463  189583.333333
3     1464  199308.333333
4     1465  213377.250000
...    ...            ...
1454  2915   88677.777778
1455  2916   88677.777778
1456  2917  142169.736842
1457  2918   95184.210526
1458  2919  290440.185185

[1459 rows x 2 columns]


Manual tuning

In [18]:
dt_model_tune1 = DecisionTreeRegressor(max_depth=20, min_samples_split=15, min_samples_leaf=5)

In [19]:
dec_tr_model2, train_loss, val_loss = split_train_evaluate(dt_model_tune1, X_train, train_target)

Training Loss: 0.11842513890828851
Validation Loss: 0.19676848651034443


In [20]:
tr_preds2 = dec_tr_model2.predict(X_train)
tr_preds2

array([204995.45454545, 192842.85714286, 205154.        , ...,
       290512.64285714, 131170.83333333, 147853.84615385], shape=(1460,))

In [21]:
test_tr_preds2 = dec_tr_model2.predict(X_test)

create_submission_file(test_tr_preds2, 'decision_tree_tuned2_sub')

        Id      SalePrice
0     1461  123241.666667
1     1462  147853.846154
2     1463  173452.846154
3     1464  172283.333333
4     1465  160900.000000
...    ...            ...
1454  2915   79136.363636
1455  2916   85910.000000
1456  2917  133583.333333
1457  2918  103833.333333
1458  2919  269756.416667

[1459 rows x 2 columns]


### Train a Random Forest Model

In [22]:
from sklearn.ensemble import RandomForestRegressor

In [23]:
rf_model = RandomForestRegressor()

In [24]:
# Train a base Random Forest model
rf_base_model, train_loss, val_loss = split_train_evaluate(rf_model, X_train, train_target)

d:\Documents\Coding Prac\ML Projects\Predicting House Prices\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
d:\Documents\Coding Prac\ML Projects\Predicting House Prices\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
d:\Documents\Coding Prac\ML Projects\Predicting House Prices\.venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
d:\Documents\Coding Prac\ML Projects\Predicting House Prices\.venv\Lib

Training Loss: 0.063393886736431
Validation Loss: 0.1504063636888347


In [25]:
rf_base_mod_preds = rf_base_model.predict(X_test)

create_submission_file(rf_base_mod_preds, 'rf_base_sub')

        Id  SalePrice
0     1461  126569.99
1     1462  158524.50
2     1463  183744.50
3     1464  182188.90
4     1465  182391.15
...    ...        ...
1454  2915   84575.93
1455  2916   87710.00
1456  2917  154974.88
1457  2918  115330.16
1458  2919  240308.60

[1459 rows x 2 columns]


### Hyperparameter Tuning Random Forests

In [26]:
rf_tune_model = RandomForestRegressor(random_state=42)

In [27]:
rf_param_grid = {
  'n_estimators': [100, 200, 300], 
  'max_depth': [10, 20, 30], 
  'min_samples_split': [2, 5, 10], 
  'min_samples_leaf': [1, 2, 5], 
  'max_features': [0.5, 0.7, 1.0]
}

In [28]:
rf_grid_search = GridSearchCV(
  estimator=rf_tune_model, 
  param_grid=rf_param_grid, 
  cv=3, 
  scoring='neg_root_mean_squared_log_error', 
  n_jobs=-1
)

In [29]:
rf_grid_search.fit(X_train, train_target)

KeyboardInterrupt: 

The model took about 30 min to train

In [ ]:
rf_grid_search.best_params_

{'max_depth': 20,
 'max_features': 0.5,
 'min_samples_leaf': 1,
 'min_samples_split': 5,
 'n_estimators': 300}

In [ ]:
rf_grid_search.best_score_

np.float64(-0.15138018444403864)

In [ ]:
best_rf_tuned_model = rf_grid_search.best_estimator_

rf_tuned_test_preds = best_rf_tuned_model.predict(X_test)
rf_tuned_test_preds

array([126661.03213792, 152362.0135648 , 187347.85323504, ...,
       155118.63814683, 113260.72827641, 226590.60444769], shape=(1459,))

In [ ]:
create_submission_file(rf_tuned_test_preds, 'rf_tuned_sub')

        Id      SalePrice
0     1461  126661.032138
1     1462  152362.013565
2     1463  187347.853235
3     1464  185705.525955
4     1465  194134.758406
...    ...            ...
1454  2915   85331.656566
1455  2916   89034.509675
1456  2917  155118.638147
1457  2918  113260.728276
1458  2919  226590.604448

[1459 rows x 2 columns]
